# F1 Multiview - Machine Learning Estimator

**Before running:** make sure the kernel (top-right) is
`Python 3.14 (.venv - F1 Multiview)`. If it is anything else
(Python 3.13, Anaconda base), `import fastf1` will fail with
`ModuleNotFoundError` even though the package is installed.

## 1. Environment check

Run this first. It proves the notebook is on the right interpreter.

In [2]:
import sys, pathlib

print("executable :", sys.executable)
print("version    :", sys.version.split()[0])

expected = pathlib.Path(".venv/Scripts/python.exe").resolve()
actual = pathlib.Path(sys.executable).resolve()

if actual == expected:
    print("OK - running inside the project .venv")
else:
    # Was just a printed warning before - easy to miss in a wall of output,
    # and missing it is exactly what caused 2026-09-07's silent
    # "Failed to load timing data!" failures (wrong kernel had pandas 3.0.3,
    # which fastf1 doesn't support). Raising stops the notebook here instead.
    raise RuntimeError(
        f"WRONG KERNEL - expected {expected}, got {actual}. "
        "Fix: Ctrl+Shift+P -> 'Notebook: Select Kernel' -> Python Environments -> .venv, "
        "then re-run this cell before continuing."
    )

executable : d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Scripts\python.exe
version    : 3.14.3
OK - running inside the project .venv


## 2. Verify fastf1 and the data stack

In [3]:
import fastf1
import fastf1.plotting
import pandas as pd
import numpy as np
import matplotlib
import scipy
import pyarrow

for name, mod in [("fastf1", fastf1), ("pandas", pd), ("numpy", np),
                  ("matplotlib", matplotlib), ("scipy", scipy), ("pyarrow", pyarrow)]:
    print(f"{name:<12} {mod.__version__}")

# Belt-and-suspenders on top of the kernel check above: fastf1 3.8.3
# declares pandas<3.0.0 and silently breaks in confusing ways under 3.x
# (this is what caused 2026-09-07's "Failed to load timing data!" run) -
# catch it here explicitly instead of hitting that failure mode again.
_pd_major = int(pd.__version__.split(".")[0])
if _pd_major >= 3:
    raise RuntimeError(
        f"pandas {pd.__version__} is installed, but fastf1 3.8.3 requires pandas<3.0.0. "
        'Fix: %pip install "pandas>=2.1.1,<3.0.0" --upgrade --quiet, then restart the kernel.'
    )

fastf1       3.8.3
pandas       2.3.3
numpy        2.5.3
matplotlib   3.11.1
scipy        1.18.1
pyarrow      25.0.1


## 3. Enable the fastf1 cache

Caching is required in practice - without it every session load re-downloads several MB from the F1 API.

In [4]:
from pathlib import Path
import fastf1

# matches the fastf1_cache/ folder already populated by today's runs -
# pointing at "cache" instead would start a second, empty cache from scratch
CACHE_DIR = Path("fastf1_cache")
CACHE_DIR.mkdir(exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))
print("cache enabled at:", CACHE_DIR.resolve())

cache enabled at: D:\Personal\Projects\F1 Multiview with Ml feed Recomender\fastf1_cache


## 4. Stage 1 — load and save raw session data

Loops over all 88 target sessions (2023-2025, every GP plus Sprint races only)
and saves each one's raw `session.laps` and `stream_data` straight to
`data/raw/` as Parquet, no stitching/cleaning applied yet. Deliberately
decoupled from stitching so the fetch (which needs multiple sittings against
FastF1's 500-requests/hour limit) never has to be redone just because the
stitching logic changes later.

Restart the kernel before running this, then run top to bottom. Safe to
stop and re-run: anything already saved gets skipped, and a rate-limit hit
stops the loop cleanly rather than crashing partway through a save.

In [5]:
import time
import logging
import traceback

import pandas as pd
import fastf1
import fastf1.api as api
import fastf1.exceptions

# was ERROR - that was silently hiding the real warning FastF1 logs when a
# single data type (e.g. laps) fails to load internally without raising,
# which is exactly what caused today's DataNotLoadedError. WARNING surfaces
# that reason instead of just the downstream symptom.
logging.getLogger("fastf1").setLevel(logging.WARNING)

RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

YEARS = (2023, 2024, 2025)


def build_race_list():
    races = []
    for year in YEARS:
        sched = fastf1.get_event_schedule(year, include_testing=False)
        for _, row in sched.iterrows():
            races.append((year, int(row["RoundNumber"]), "R", row["EventName"]))
            if row["EventFormat"] != "conventional":
                races.append((year, int(row["RoundNumber"]), "S", row["EventName"]))
    return races


def race_slug(year, rnd, code, event_name):
    slug = event_name.lower().replace("grand prix", "").strip().replace(" ", "_")
    return f"{year}_{rnd:02d}_{slug}_{code}"


races = build_race_list()
print(f"{len(races)} sessions queued")

d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Lib\site-packages\fastf1\api.py:32: UserWarning: `fastf1.api` will be considered private in future releases and potentially be removed or changed!
  warnings.warn("`fastf1.api` will be considered private in future releases and "


88 sessions queued


In [5]:
def load_and_save_raw(year, rnd, code, max_retries=3):
    """Fetch one session's raw laps/stream data, retrying with backoff on
    DataNotLoadedError - that error means session.load() ran without
    raising but one piece (laps or telemetry) silently didn't finish.
    2026-09-07's real cause turned out to be a wrong-kernel/pandas-version
    problem, not something a retry could fix, so this is defensive: it
    covers genuine transient hiccups without pretending to fix a
    persistent, environment-level failure (the loop's circuit breaker
    below handles that case instead)."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            session = fastf1.get_session(year, rnd, code)
            session.load(laps=True, telemetry=True, weather=False, messages=True)

            # session.laps is FastF1's own `Laps` subclass - wrapping in a
            # fresh pd.DataFrame() here is just normalizing the type, not
            # related to the earlier parquet-write bug (that turned out to
            # be the pandas/pyarrow version mismatch, now fixed).
            laps_data = pd.DataFrame(session.laps)
            _, stream_data = api.timing_data(session.api_path)
            stream_data = pd.DataFrame(stream_data)
            return laps_data, stream_data

        except fastf1.exceptions.DataNotLoadedError as e:
            last_err = e
            wait = 5 * attempt  # 5s, 10s, 15s - backs off instead of hammering
            print(f"(attempt {attempt}/{max_retries}: didn't fully load, waiting {wait}s) ", end="", flush=True)
            time.sleep(wait)

    raise last_err

In [9]:
YEAR = 2022
ROUND = 21
SESSION_CODE = "R"

RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def race_slug(year, rnd, code, event_name):
    import re
    slug = re.sub(r"[^a-z0-9]+", "_", event_name.lower()).strip("_")
    return f"{year}_{rnd:02d}_{slug}_{code}"

def combine_telemetry(data_dict, driver_map, kind):
    """data_dict: {driver_number(str): DataFrame} from session.car_data / session.pos_data"""
    frames = []
    for drv_num, tel in data_dict.items():
        df = pd.DataFrame(tel)
        df.insert(0, "DriverNumber", str(drv_num))
        frames.append(df)
    if not frames:
        print(f"      WARNING: no {kind} for any driver")
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.merge(driver_map, on="DriverNumber", how="left")
    if combined["Driver"].isna().any():
        missing = combined.loc[combined["Driver"].isna(), "DriverNumber"].unique().tolist()
        print(f"      WARNING: {kind} has driver numbers with no abbreviation match: {missing}")
    return combined

print("\n" + "=" * 70)
t_start = time.time()

try:
    print(f"[1/7] fastf1.get_session({YEAR}, {ROUND}, '{SESSION_CODE}')")
    t0 = time.time()
    session = fastf1.get_session(YEAR, ROUND, SESSION_CODE)
    event_name = session.event["EventName"]
    name = race_slug(YEAR, ROUND, SESSION_CODE, event_name)
    print(f"      -> {event_name}  ({time.time() - t0:.2f}s)  file stem: {name}")

    print("\n[2/7] session.load(laps=True, telemetry=True, weather=False, messages=True)")
    t0 = time.time()
    session.load(laps=True, telemetry=True, weather=False, messages=True)
    print(f"      -> done ({time.time() - t0:.2f}s)")

    print("\n[3/7] session.laps (processed)")
    t0 = time.time()
    laps_data = pd.DataFrame(session.laps)
    print(f"      -> {len(laps_data)} rows, {len(laps_data.columns)} cols ({time.time() - t0:.2f}s)")
    if laps_data.empty:
        print("      WARNING: empty, load() silently failed for laps")

    driver_map = laps_data[["DriverNumber", "Driver"]].drop_duplicates()
    driver_map["DriverNumber"] = driver_map["DriverNumber"].astype(str)

    print("\n[4/7] api.timing_data(session.api_path)  (raw laps + stream, should be a cache-hit after step 2)")
    t0 = time.time()
    laps_raw_data, stream_data = api.timing_data(session.api_path)
    laps_raw_data = pd.DataFrame(laps_raw_data)
    stream_data = pd.DataFrame(stream_data)
    print(f"      -> laps_raw: {len(laps_raw_data)} rows, {len(laps_raw_data.columns)} cols")
    print(f"      -> stream  : {len(stream_data)} rows, {len(stream_data.columns)} cols  ({time.time() - t0:.2f}s)")

    print("\n[5/7] session.car_data (per driver, concatenating)")
    t0 = time.time()
    car_data = combine_telemetry(session.car_data, driver_map, "car_data")
    n_drv = car_data["DriverNumber"].nunique() if not car_data.empty else 0
    print(f"      -> {len(car_data)} rows, {len(car_data.columns)} cols, {n_drv} drivers ({time.time() - t0:.2f}s)")

    print("\n[6/7] session.pos_data (per driver, concatenating)")
    t0 = time.time()
    pos_data = combine_telemetry(session.pos_data, driver_map, "pos_data")
    n_drv = pos_data["DriverNumber"].nunique() if not pos_data.empty else 0
    print(f"      -> {len(pos_data)} rows, {len(pos_data.columns)} cols, {n_drv} drivers ({time.time() - t0:.2f}s)")

    print("\n[7/7] writing parquet files")
    t0 = time.time()
    outputs = {
        "laps": laps_data,
        "laps_raw": laps_raw_data,
        "stream": stream_data,
        "car": car_data,
        "pos": pos_data,
    }
    for tag, df in outputs.items():
        path = RAW_DIR / f"{name}_{tag}.parquet"
        df.to_parquet(path)
        print(f"      -> {path}  ({path.stat().st_size:,} bytes)")
    print(f"      ({time.time() - t0:.2f}s)")

    print("\n" + "=" * 70)
    print(f"SUCCESS - {name}  total {time.time() - t_start:.1f}s")
    print("=" * 70)

except Exception as e:
    print("\n" + "=" * 70)
    print(f"FAILED - {type(e).__name__}: {e}")
    print("=" * 70)
    traceback.print_exc()


[1/7] fastf1.get_session(2022, 21, 'R')
      -> São Paulo Grand Prix  (0.02s)  file stem: 2022_21_s_o_paulo_grand_prix_R

[2/7] session.load(laps=True, telemetry=True, weather=False, messages=True)


_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


      -> done (26.63s)

[3/7] session.laps (processed)
      -> 1259 rows, 31 cols (0.00s)

[4/7] api.timing_data(session.api_path)  (raw laps + stream, should be a cache-hit after step 2)
      -> laps_raw: 1256 rows, 18 cols
      -> stream  : 24169 rows, 5 cols  (0.01s)

[5/7] session.car_data (per driver, concatenating)
      -> 750160 rows, 12 cols, 20 drivers (0.15s)

[6/7] session.pos_data (per driver, concatenating)
      -> 766180 rows, 10 cols, 20 drivers (0.23s)

[7/7] writing parquet files
      -> data\raw\2022_21_s_o_paulo_grand_prix_R_laps.parquet  (128,800 bytes)
      -> data\raw\2022_21_s_o_paulo_grand_prix_R_laps_raw.parquet  (97,038 bytes)
      -> data\raw\2022_21_s_o_paulo_grand_prix_R_stream.parquet  (431,671 bytes)
      -> data\raw\2022_21_s_o_paulo_grand_prix_R_car.parquet  (6,942,501 bytes)
      -> data\raw\2022_21_s_o_paulo_grand_prix_R_pos.parquet  (7,407,444 bytes)
      (0.86s)

SUCCESS - 2022_21_s_o_paulo_grand_prix_R  total 27.9s


In [16]:
# =========================================================
# Stage 1 batch loader: 2024 season, round 8 onward
# Loops races automatically. On error, logs FAILED and moves to the
# next race, no early stop. Captures every WARNING+ record fastf1
# emits per race into a structured log, so results can be reviewed
# without pasting console output.
# =========================================================

import sys
import time
import logging
from pathlib import Path

import fastf1
from fastf1 import api
import pandas as pd

# --- 0. environment check ---
print(f"python : {sys.executable}")
print(f"pandas : {pd.__version__}  fastf1 : {fastf1.__version__}")
if int(pd.__version__.split(".")[0]) >= 3:
    raise RuntimeError(f"pandas {pd.__version__} >= 3.0, wrong kernel? fastf1 3.8.3 needs pandas<3.0.0")

RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW_DIR / "_load_log_2022.csv"

def race_slug(year, rnd, code, event_name):
    import re
    slug = re.sub(r"[^a-z0-9]+", "_", event_name.lower()).strip("_")
    return f"{year}_{rnd:02d}_{slug}_{code}"

# --- 1. build the race list straight from fastf1's schedule, same source Race Calendar.md was generated from ---
YEAR = 2022
START_ROUND = 1

sched = fastf1.get_event_schedule(YEAR, include_testing=False).sort_values("RoundNumber")
races = []
for _, row in sched.iterrows():
    rnd = int(row["RoundNumber"])
    if rnd < START_ROUND:
        continue
    event_name = row["EventName"]
    session_names = [row.get(f"Session{i}") for i in range(1, 6)]
    has_sprint = any(isinstance(s, str) and "sprint" in s.lower() for s in session_names)
    races.append((YEAR, rnd, "R", event_name))
    if has_sprint:
        races.append((YEAR, rnd, "S", event_name))

print(f"\n{len(races)} sessions queued, {YEAR} round {START_ROUND} onward\n")

# --- 2. capture every WARNING+ record fastf1 emits, per race, instead of relying on console output ---
class CaptureHandler(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.records = []
    def emit(self, record):
        self.records.append(self.format(record))
    def reset(self):
        self.records = []

capture = CaptureHandler()
capture.setFormatter(logging.Formatter("%(name)s %(levelname)s %(message)s"))
fastf1_logger = logging.getLogger("fastf1")
fastf1_logger.setLevel(logging.WARNING)
fastf1_logger.addHandler(capture)

def combine_telemetry(data_dict, driver_map):
    frames = []
    for drv_num, tel in data_dict.items():
        df = pd.DataFrame(tel)
        df.insert(0, "DriverNumber", str(drv_num))
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).merge(driver_map, on="DriverNumber", how="left")

def load_one_race(year, rnd, code, max_retries=3):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            session = fastf1.get_session(year, rnd, code)
            session.load(laps=True, telemetry=True, weather=False, messages=True)

            laps_data = pd.DataFrame(session.laps)
            driver_map = laps_data[["DriverNumber", "Driver"]].drop_duplicates()
            driver_map["DriverNumber"] = driver_map["DriverNumber"].astype(str)

            laps_raw_data, stream_data = api.timing_data(session.api_path)
            laps_raw_data = pd.DataFrame(laps_raw_data)
            stream_data = pd.DataFrame(stream_data)

            car_data = combine_telemetry(session.car_data, driver_map)
            pos_data = combine_telemetry(session.pos_data, driver_map)

            return {
                "laps": laps_data, "laps_raw": laps_raw_data,
                "stream": stream_data, "car": car_data, "pos": pos_data,
            }
        except fastf1.exceptions.DataNotLoadedError as e:
            last_err = e
            wait = 5 * attempt
            print(f"  (attempt {attempt}/{max_retries}, waiting {wait}s)")
            time.sleep(wait)
    raise last_err

# --- 3. run the loop: skip failures, keep going, log everything after every race ---
results = []

for i, (year, rnd, code, event_name) in enumerate(races, 1):
    name = race_slug(year, rnd, code, event_name)
    print(f"[{i}/{len(races)}] {name}...", end=" ", flush=True)

    if len(list(RAW_DIR.glob(f"{name}_*.parquet"))) >= 5:
        print("SKIP (already saved)")
        continue

    capture.reset()
    t0 = time.time()
    row = {"race": name, "year": year, "round": rnd, "session": code, "event": event_name}
    try:
        outputs = load_one_race(year, rnd, code)
        for tag, df in outputs.items():
            df.to_parquet(RAW_DIR / f"{name}_{tag}.parquet")
        row.update({
            "status": "OK", "laps_rows": len(outputs["laps"]), "stream_rows": len(outputs["stream"]),
            "seconds": round(time.time() - t0, 1), "warnings": " | ".join(capture.records), "error": "",
        })
        print(f"OK ({time.time() - t0:.1f}s, {len(capture.records)} warnings)")
    except Exception as e:
        row.update({
            "status": "FAILED", "laps_rows": None, "stream_rows": None,
            "seconds": round(time.time() - t0, 1), "warnings": " | ".join(capture.records),
            "error": f"{type(e).__name__}: {e}",
        })
        print(f"FAILED: {type(e).__name__}: {e}")

    results.append(row)
    pd.DataFrame(results).to_csv(LOG_PATH, index=False)  # write after every race, never lose progress

print(f"\nDone this session. {sum(r['status']=='OK' for r in results)}/{len(results)} succeeded.")
print(f"Full structured log: {LOG_PATH}")

python : d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Scripts\python.exe
pandas : 2.3.3  fastf1 : 3.8.3

25 sessions queued, 2022 round 1 onward

[1/25] 2022_01_bahrain_grand_prix_R... 

core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!


OK (57.6s, 3 warnings)
[2/25] 2022_02_saudi_arabian_grand_prix_R... 

core        WARNING 	No lap data for driver 22
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!


OK (73.5s, 5 warnings)
[3/25] 2022_03_australian_grand_prix_R... 

...

OK (51.5s, 3 warnings)
[20/25] 2022_18_japanese_grand_prix_R... 

_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


OK (47.1s, 3 warnings)
[21/25] 2022_19_united_states_grand_prix_R... 

_api        WARNING 	Driver 10: Ignoring late data for a previously processed lap.The data may contain errors (previous: 6; current 7)
_api        WARNING 	Driver 20: Ignoring late data for a previously processed lap.The data may contain errors (previous: 31; current 32)
_api        WARNING 	Failed to align laps for drivers: ['55']
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


OK (41.8s, 6 warnings)
[22/25] 2022_20_mexico_city_grand_prix_R... OK (37.6s, 0 warnings)
[23/25] 2022_21_s_o_paulo_grand_prix_R... FAILED: RateLimitExceededError: any API: 500 calls/h
[24/25] 2022_21_s_o_paulo_grand_prix_S... FAILED: RateLimitExceededError: any API: 500 calls/h
[25/25] 2022_22_abu_dhabi_grand_prix_R... FAILED: RateLimitExceededError: any API: 500 calls/h

Done this session. 22/25 succeeded.
Full structured log: data\raw\_load_log_2022.csv


In [1]:
# =========================================================
# Diagnostic-only: 2023 Rounds 2-6, R sessions
# Full traceback capture, no early stop, no rate-limit skip logic
# needed (only 5 races). Writes everything to a text log file.
# =========================================================

import sys
import time
import logging
import traceback
from pathlib import Path

import fastf1
from fastf1 import api
import pandas as pd

print(f"python : {sys.executable}")
print(f"pandas : {pd.__version__}  fastf1 : {fastf1.__version__}")
if int(pd.__version__.split(".")[0]) >= 3:
    raise RuntimeError(f"pandas {pd.__version__} >= 3.0, wrong kernel? fastf1 3.8.3 needs pandas<3.0.0")

RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW_DIR / "_diagnostic_log_2023_r2-6.txt"

# capture every WARNING+ record fastf1 emits, per race
class CaptureHandler(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.records = []
    def emit(self, record):
        self.records.append(self.format(record))
    def reset(self):
        self.records = []

capture = CaptureHandler()
capture.setFormatter(logging.Formatter("%(name)s %(levelname)s %(message)s"))
fastf1_logger = logging.getLogger("fastf1")
fastf1_logger.setLevel(logging.WARNING)
fastf1_logger.addHandler(capture)

RACES = [
    (2023, 2, "R", "Saudi Arabian Grand Prix"),
    (2023, 3, "R", "Australian Grand Prix"),
    (2023, 4, "R", "Azerbaijan Grand Prix"),
    (2023, 5, "R", "Miami Grand Prix"),
    (2023, 6, "R", "Monaco Grand Prix"),
]

log_lines = []

def log(msg):
    print(msg)
    log_lines.append(msg)

for year, rnd, code, expected_name in RACES:
    log("\n" + "=" * 70)
    log(f"{year} Round {rnd} ({expected_name}), {code}")
    log("=" * 70)
    capture.reset()
    t0 = time.time()

    try:
        session = fastf1.get_session(year, rnd, code)
        log(f"[1/5] get_session OK -> {session.event['EventName']}  ({time.time()-t0:.2f}s)")

        t1 = time.time()
        session.load(laps=True, telemetry=True, weather=False, messages=True)
        log(f"[2/5] session.load OK  ({time.time()-t1:.2f}s)")

        t1 = time.time()
        laps_data = pd.DataFrame(session.laps)
        log(f"[3/5] session.laps -> {len(laps_data)} rows, {len(laps_data.columns)} cols  ({time.time()-t1:.2f}s)")

        t1 = time.time()
        laps_raw_data, stream_data = api.timing_data(session.api_path)
        laps_raw_data = pd.DataFrame(laps_raw_data)
        stream_data = pd.DataFrame(stream_data)
        log(f"[4/5] api.timing_data OK -> laps_raw {len(laps_raw_data)} rows, stream {len(stream_data)} rows  ({time.time()-t1:.2f}s)")

        t1 = time.time()
        n_car = len(session.car_data)
        n_pos = len(session.pos_data)
        log(f"[5/5] car_data: {n_car} drivers, pos_data: {n_pos} drivers  ({time.time()-t1:.2f}s)")

        log(f"\nSUCCESS, total {time.time()-t0:.1f}s")
        if capture.records:
            log(f"\n{len(capture.records)} warnings during load:")
            for r in capture.records:
                log(f"  {r}")
        else:
            log("\nzero warnings")

    except Exception as e:
        log(f"\nFAILED after {time.time()-t0:.1f}s: {type(e).__name__}: {e}")
        if capture.records:
            log(f"\n{len(capture.records)} warnings before the failure:")
            for r in capture.records:
                log(f"  {r}")
        log("\n--- FULL TRACEBACK ---")
        log(traceback.format_exc())

LOG_PATH.write_text("\n".join(log_lines), encoding="utf-8")
print(f"\n\nFull diagnostic log written to: {LOG_PATH}")                                        

d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Lib\site-packages\fastf1\api.py:32: UserWarning: `fastf1.api` will be considered private in future releases and potentially be removed or changed!
  warnings.warn("`fastf1.api` will be considered private in future releases and "
req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\fastf1


python : d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Scripts\python.exe
pandas : 2.3.3  fastf1 : 3.8.3

2023 Round 2 (Saudi Arabian Grand Prix), R
[1/5] get_session OK -> Saudi Arabian Grand Prix  (1.15s)


logger      WARNING 	Failed to load session info data!
core        WARNING 	Failed to load extended driver information!
logger      WARNING 	Failed to load session status data!
logger      WARNING 	Failed to load total lap count!
logger      WARNING 	Failed to load track status data!
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed to add first lap time from Ergast for drivers: ['14', '11', '63', '18', '55', '31', '44', '10', '16', '24', '20', '27', '1', '22', '77', '23', '4', '21', '81', '2']
logger      WARNING 	Failed to load telemetry data!
logger      WARNING 	Failed to load race control messages!


[2/5] session.load OK  (4.06s)

FAILED after 5.2s: DataNotLoadedError: The data you are trying to access has not been loaded yet. See `Session.load`

10 warnings before the failure:
  fastf1.fastf1.req WARNING DEFAULT CACHE ENABLED! (24.0 KB) C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\fastf1
  fastf1.fastf1.core WARNING Failed to load session info data!
  fastf1.fastf1.core WARNING Failed to load extended driver information!
  fastf1.fastf1.core WARNING Failed to load session status data!
  fastf1.fastf1.core WARNING Failed to load total lap count!
  fastf1.fastf1.core WARNING Failed to load track status data!
  fastf1.fastf1.core WARNING Failed to load timing data!
  fastf1.fastf1.core WARNING Failed to add first lap time from Ergast for drivers: ['14', '11', '63', '18', '55', '31', '44', '10', '16', '24', '20', '27', '1', '22', '77', '23', '4', '21', '81', '2']
  fastf1.fastf1.core WARNING Failed to load telemetry data!
  fastf1.fastf1.core WARNING Failed to load race control messa

logger      WARNING 	Failed to load session info data!
core        WARNING 	Failed to load extended driver information!
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


[2/5] session.load OK  (65.19s)
[3/5] session.laps -> 1003 rows, 31 cols  (0.00s)
[4/5] api.timing_data OK -> laps_raw 995 rows, stream 21024 rows  (0.02s)
[5/5] car_data: 20 drivers, pos_data: 20 drivers  (0.00s)

SUCCESS, total 65.2s

5 warnings during load:
  fastf1.fastf1.core WARNING Failed to load session info data!
  fastf1.fastf1.core WARNING Failed to load extended driver information!
  fastf1.api WARNING Driver 241: Position data is incomplete!
  fastf1.api WARNING Driver 242: Position data is incomplete!
  fastf1.api WARNING Driver 243: Position data is incomplete!

2023 Round 4 (Azerbaijan Grand Prix), R
[1/5] get_session OK -> Azerbaijan Grand Prix  (0.02s)


_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


[2/5] session.load OK  (30.98s)
[3/5] session.laps -> 962 rows, 31 cols  (0.00s)
[4/5] api.timing_data OK -> laps_raw 961 rows, stream 23107 rows  (0.02s)
[5/5] car_data: 20 drivers, pos_data: 20 drivers  (0.00s)

SUCCESS, total 31.0s

3 warnings during load:
  fastf1.api WARNING Driver 241: Position data is incomplete!
  fastf1.api WARNING Driver 242: Position data is incomplete!
  fastf1.api WARNING Driver 243: Position data is incomplete!

2023 Round 5 (Miami Grand Prix), R
[1/5] get_session OK -> Miami Grand Prix  (0.02s)
[2/5] session.load OK  (31.85s)
[3/5] session.laps -> 1138 rows, 31 cols  (0.00s)
[4/5] api.timing_data OK -> laps_raw 1138 rows, stream 28025 rows  (0.02s)
[5/5] car_data: 20 drivers, pos_data: 20 drivers  (0.00s)

SUCCESS, total 31.9s

zero warnings

2023 Round 6 (Monaco Grand Prix), R
[1/5] get_session OK -> Monaco Grand Prix  (0.02s)
[2/5] session.load OK  (34.67s)
[3/5] session.laps -> 1515 rows, 31 cols  (0.00s)
[4/5] api.timing_data OK -> laps_raw 1514 rows

In [3]:
# =========================================================
# Stage 2 + Stage 3 validation (FIXED — see note below)
# Races: 2021 Round 1 (Bahrain), Round 2 (Emilia Romagna)
# Reads data/raw/, builds the per-driver 5s grid, aligns stream_data
# (with the gap-parsing rule applied BEFORE the join), telemetry, and
# lap context onto it via merge_asof. Includes sanity checks, not just
# the join itself. Saves stitched output to data/stitched/.
#
# FIX (2026-09-10): stream_data's Driver column holds raw car-number
# strings ('10', '44', ...), NOT 3-letter abbreviations like laps_data
# and car_data use ('GAS', 'HAM', ...). This caused merge_asof(by="Driver")
# to match nothing, so gap_ahead/gap_leader/Position came back 100% NaN
# in every row of both races. Fix: remap stream_df['Driver'] from car
# number -> abbreviation (using that race's own laps_df) immediately
# after loading, before anything else touches it.
# =========================================================

import re
import sys
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

print(f"python : {sys.executable}")
print(f"pandas : {pd.__version__}")
if int(pd.__version__.split(".")[0]) >= 3:
    raise RuntimeError(f"pandas {pd.__version__} >= 3.0, wrong kernel?")

RAW_DIR = Path("data") / "raw"
STITCHED_DIR = Path("data") / "stitched"
STITCHED_DIR.mkdir(parents=True, exist_ok=True)

RACES = [
    "2021_01_bahrain_grand_prix_R",
    "2021_02_emilia_romagna_grand_prix_R",
]
GRID_FREQ = "5s"

def parse_gap_value(raw, position):
    """Four-shape parsing rule for GapToLeader / IntervalToPositionAhead."""
    if pd.isna(raw):
        return np.nan, "missing"
    s = str(raw).strip()
    if re.fullmatch(r"[+-]\d+\.?\d*", s):               # shape 1: real gap
        return float(s), "real"
    if re.fullmatch(r"LAP \d+", s):                       # shape 2: lap-crossing marker
        return (0.0 if position == 1 else np.nan), "lap_marker"
    if re.fullmatch(r"\d+ L", s):                          # shape 3: genuine lapped status
        return np.nan, "lapped"
    if re.fullmatch(r"\d+L", s):                           # shape 4: malformed, session-end artifact
        return np.nan, "malformed"
    return np.nan, "malformed"

def clean_gap_column(df, raw_col, out_col):
    """Clean at the stream's own native timestamps, BEFORE merge_asof ever sees it."""
    parsed = df.apply(lambda r: parse_gap_value(r[raw_col], r["Position"]), axis=1)
    df[out_col] = [p[0] for p in parsed]
    df[f"{out_col}_kind"] = [p[1] for p in parsed]
    df[f"{out_col}_is_lapped"] = df[f"{out_col}_kind"] == "lapped"
    df[out_col] = df.groupby("Driver")[out_col].ffill()
    return df

def fix_stream_driver_ids(stream_df, laps_df):
    """FIX: stream_df['Driver'] is car-number strings; remap to the same
    3-letter abbreviation scheme laps_df/car_df use, via laps_df's own
    DriverNumber -> Driver mapping (safest: built fresh per race)."""
    num_to_abbr = dict(laps_df[["DriverNumber", "Driver"]].drop_duplicates().values)
    stream_df = stream_df.copy()
    stream_df["Driver"] = stream_df["Driver"].map(num_to_abbr)
    unmapped = stream_df["Driver"].isna().sum()
    if unmapped:
        print(f"    WARNING: {unmapped}/{len(stream_df)} stream_data rows had a car number "
              f"not found in this race's laps_data — dropping them")
        stream_df = stream_df.dropna(subset=["Driver"])
    return stream_df

def build_grid(laps_df, freq=GRID_FREQ):
    """Stage 2: one row per driver per 5s step, each driver's own start/end bounds."""
    grids = []
    for driver, laps in laps_df.groupby("Driver"):
        start = laps["LapStartTime"].min()
        end = laps["Time"].max()
        if pd.isna(start) or pd.isna(end) or end <= start:
            print(f"    WARNING: skipping {driver}, bad bounds (start={start}, end={end})")
            continue
        times = pd.timedelta_range(start=start, end=end, freq=freq)
        grids.append(pd.DataFrame({"Driver": driver, "Time": times}))
    return pd.concat(grids, ignore_index=True).sort_values(["Driver", "Time"]).reset_index(drop=True)

def stitch_race(name):
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    laps_df = pd.read_parquet(RAW_DIR / f"{name}_laps.parquet")
    stream_df = pd.read_parquet(RAW_DIR / f"{name}_stream.parquet")
    car_df = pd.read_parquet(RAW_DIR / f"{name}_car.parquet")

    print(f"[load] laps: {len(laps_df)} rows, stream: {len(stream_df)} rows, car: {len(car_df)} rows")
    print(f"       drivers -> laps: {laps_df['Driver'].nunique()}, stream (raw, car numbers): {stream_df['Driver'].nunique()}, car: {car_df['Driver'].nunique()}")

    # ---- FIX: remap stream_data's Driver from car number -> abbreviation ----
    stream_df = fix_stream_driver_ids(stream_df, laps_df)
    print(f"       stream drivers after remap: {stream_df['Driver'].nunique()} ({sorted(stream_df['Driver'].unique())})")

    # ---- Stage 2 ----
    grid = build_grid(laps_df)
    print(f"[grid] {len(grid)} rows, {grid['Driver'].nunique()} drivers")

    span = grid.groupby("Driver")["Time"].agg(["min", "max", "count"])
    session_end = span["max"].max()
    early_cutoff = session_end - pd.Timedelta(minutes=5)
    retired = span[span["max"] < early_cutoff]
    if len(retired):
        print(f"[check] retirement-cutoff: {len(retired)} driver(s) whose grid ends >5min before session end:")
        print(retired)
    else:
        print("[check] retirement-cutoff: no driver's grid ends notably early - no obvious retirement here")

    # ---- Stage 3a: clean stream_data BEFORE aligning ----
    stream_df = stream_df.sort_values(["Driver", "Time"]).reset_index(drop=True)
    stream_df = clean_gap_column(stream_df, "IntervalToPositionAhead", "gap_ahead")
    stream_df = clean_gap_column(stream_df, "GapToLeader", "gap_leader")
    print(f"[parse] IntervalToPositionAhead shapes: {dict(stream_df['gap_ahead_kind'].value_counts())}")
    print(f"[parse] gap_ahead still NaN after forward-fill: {stream_df['gap_ahead'].isna().sum()} / {len(stream_df)} "
          f"(expected: each driver's first sample before any real reading arrives)")

    # ---- Stage 3b/c/d: merge_asof onto the grid ----
    stream_cols = ["Driver", "Time", "Position", "gap_ahead", "gap_ahead_is_lapped", "gap_leader", "gap_leader_is_lapped"]
    stitched = pd.merge_asof(grid.sort_values("Time"), stream_df[stream_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    car_cols = ["Driver", "Time", "Speed", "DRS"]
    stitched = pd.merge_asof(stitched.sort_values("Time"), car_df[car_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    laps_join = laps_df.rename(columns={"Time": "_LapEndTime", "LapStartTime": "Time"})
    laps_cols = ["Driver", "Time", "LapNumber", "Compound", "TyreLife", "Stint", "PitInTime", "PitOutTime", "TrackStatus"]
    stitched = pd.merge_asof(stitched.sort_values("Time"), laps_join[laps_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    print(f"[stitch] final table: {len(stitched)} rows, {len(stitched.columns)} columns")
    print(f"         columns: {list(stitched.columns)}")

    # sanity check: gap_ahead should now be populated, not 100% NaN
    gap_nan_pct = stitched["gap_ahead"].isna().mean() * 100
    print(f"[check] gap_ahead NaN in final stitched table: {stitched['gap_ahead'].isna().sum()}/{len(stitched)} ({gap_nan_pct:.1f}%)")
    if gap_nan_pct > 5:
        print("        WARNING: still unexpectedly high - investigate further")

    # sanity check: TrackStatus (1 = green/all clear, anything else = flag/SC/VSC/red)
    print(f"[check] TrackStatus value counts:\n{stitched['TrackStatus'].value_counts(dropna=False)}")

    # sanity check: time-reference alignment on one real lap
    sample_driver = laps_df["Driver"].iloc[0]
    driver_laps = laps_df[laps_df["Driver"] == sample_driver].dropna(subset=["LapStartTime", "Time"]).reset_index(drop=True)
    sample_lap = driver_laps.iloc[len(driver_laps) // 2]
    lap_start, lap_end = sample_lap["LapStartTime"], sample_lap["Time"]
    window = stitched[(stitched["Driver"] == sample_driver) & (stitched["Time"] >= lap_start) & (stitched["Time"] <= lap_end)]
    print(f"[check] time-reference spot check - {sample_driver}, lap {sample_lap['LapNumber']}, {lap_start} -> {lap_end}:")
    print(window[["Time", "Speed", "DRS", "gap_ahead"]].to_string(index=False))
    print("        (Speed should rise/fall like a real lap - not flat, random, or from a different part of the race)")

    out_path = STITCHED_DIR / f"{name}.parquet"
    stitched.to_parquet(out_path)
    print(f"[save] {out_path} ({out_path.stat().st_size:,} bytes)")
    return stitched

results = {}
for name in RACES:
    try:
        results[name] = stitch_race(name)
    except Exception:
        print(f"\nFAILED on {name}")
        traceback.print_exc()

print("\n" + "=" * 70)
print(f"Done. {len(results)}/{len(RACES)} races stitched successfully.")
print("=" * 70)

python : d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Scripts\python.exe
pandas : 2.3.3

2021_01_bahrain_grand_prix_R
[load] laps: 1027 rows, stream: 27893 rows, car: 660800 rows
       drivers -> laps: 20, stream (raw, car numbers): 19, car: 20
       stream drivers after remap: 19 (['ALO', 'BOT', 'GAS', 'GIO', 'HAM', 'LAT', 'LEC', 'MSC', 'NOR', 'OCO', 'PER', 'RAI', 'RIC', 'RUS', 'SAI', 'STR', 'TSU', 'VER', 'VET'])
[grid] 20622 rows, 20 drivers
[check] retirement-cutoff: 4 driver(s) whose grid ends >5min before session end:
                          min                    max  count
Driver                                                     
ALO    0 days 00:37:09.970000 0 days 01:32:29.970000    665
GAS    0 days 00:37:09.970000 0 days 02:04:49.970000   1053
LAT    0 days 00:37:09.970000 0 days 02:04:19.970000   1047
MAZ    0 days 00:37:09.970000 0 days 00:39:04.970000     24


C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\ipykernel_4832\3945571184.py:114: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  early_cutoff = session_end - pd.Timedelta(minutes=5)


[parse] IntervalToPositionAhead shapes: {'real': np.int64(27765), 'lap_marker': np.int64(87), 'missing': np.int64(38), 'malformed': np.int64(3)}
[parse] gap_ahead still NaN after forward-fill: 38 / 27893 (expected: each driver's first sample before any real reading arrives)
[stitch] final table: 20622 rows, 16 columns
         columns: ['Driver', 'Time', 'Position', 'gap_ahead', 'gap_ahead_is_lapped', 'gap_leader', 'gap_leader_is_lapped', 'Speed', 'DRS', 'LapNumber', 'Compound', 'TyreLife', 'Stint', 'PitInTime', 'PitOutTime', 'TrackStatus']
[check] gap_ahead NaN in final stitched table: 53/20622 (0.3%)
[check] TrackStatus value counts:
TrackStatus
1       17770
41        557
124       542
4         534
126       386
671       348
12        348
1267       52
71         46
21         39
Name: count, dtype: int64
[check] time-reference spot check - HAM, lap 29.0, 0 days 01:24:42.938000 -> 0 days 01:26:38.132000:
                  Time  Speed  DRS  gap_ahead
0 days 01:24:44.970000   80.0  

C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\ipykernel_4832\3945571184.py:114: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  early_cutoff = session_end - pd.Timedelta(minutes=5)


[parse] IntervalToPositionAhead shapes: {'real': np.int64(22466), 'lap_marker': np.int64(74), 'missing': np.int64(40), 'malformed': np.int64(2), 'lapped': np.int64(1)}
[parse] gap_ahead still NaN after forward-fill: 40 / 22583 (expected: each driver's first sample before any real reading arrives)
[stitch] final table: 26459 rows, 16 columns
         columns: ['Driver', 'Time', 'Position', 'gap_ahead', 'gap_ahead_is_lapped', 'gap_leader', 'gap_leader_is_lapped', 'Speed', 'DRS', 'LapNumber', 'Compound', 'TyreLife', 'Stint', 'PitInTime', 'PitOutTime', 'TrackStatus']
[check] gap_ahead NaN in final stitched table: 72/26459 (0.3%)
[check] TrackStatus value counts:
TrackStatus
1      14990
51      3482
4       2355
45      2316
12      1892
124      507
41       498
24       403
21        16
Name: count, dtype: int64
[check] time-reference spot check - GAS, lap 32.0, 0 days 01:27:22.493000 -> 0 days 01:29:21.965000:
                  Time  Speed  DRS  gap_ahead
0 days 01:27:25.688000  262.0  

In [4]:
# =========================================================
# Stage 2 + Stage 3 — targeted check on lap-alignment-warning races
# Races: 2021 Round 3 (Portuguese), 2021 Round 8 (Styrian), 2022 Round 2 (Saudi Arabian)
# These three hit the "Ignoring late data" / "Failed to align laps for
# drivers" / "timing integrity error" / "No lap data for driver X" warning
# family flagged in Data Pipeline Plan.md as the one risk category not yet
# exercised by the two 2021 validation races already confirmed clean.
# Same fixed Stage 2/3 logic as before (includes the stream_data Driver
# car-number -> abbreviation remap). Reads data/raw/, writes data/stitched/.
# =========================================================

import re
import sys
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

print(f"python : {sys.executable}")
print(f"pandas : {pd.__version__}")
if int(pd.__version__.split(".")[0]) >= 3:
    raise RuntimeError(f"pandas {pd.__version__} >= 3.0, wrong kernel?")

RAW_DIR = Path("data") / "raw"
STITCHED_DIR = Path("data") / "stitched"
STITCHED_DIR.mkdir(parents=True, exist_ok=True)

RACES = [
    "2021_03_portuguese_grand_prix_R",   # "Ignoring late data" + "Failed to align laps"
    "2021_08_styrian_grand_prix_R",      # "Failed to align laps" + "timing integrity error (might be a bug)"
    "2022_02_saudi_arabian_grand_prix_R",  # "No lap data for driver 22" + "all laps inaccurate" x2
]
GRID_FREQ = "5s"

def parse_gap_value(raw, position):
    if pd.isna(raw):
        return np.nan, "missing"
    s = str(raw).strip()
    if re.fullmatch(r"[+-]\d+\.?\d*", s):
        return float(s), "real"
    if re.fullmatch(r"LAP \d+", s):
        return (0.0 if position == 1 else np.nan), "lap_marker"
    if re.fullmatch(r"\d+ L", s):
        return np.nan, "lapped"
    if re.fullmatch(r"\d+L", s):
        return np.nan, "malformed"
    return np.nan, "malformed"

def clean_gap_column(df, raw_col, out_col):
    parsed = df.apply(lambda r: parse_gap_value(r[raw_col], r["Position"]), axis=1)
    df[out_col] = [p[0] for p in parsed]
    df[f"{out_col}_kind"] = [p[1] for p in parsed]
    df[f"{out_col}_is_lapped"] = df[f"{out_col}_kind"] == "lapped"
    df[out_col] = df.groupby("Driver")[out_col].ffill()
    return df

def fix_stream_driver_ids(stream_df, laps_df):
    num_to_abbr = dict(laps_df[["DriverNumber", "Driver"]].drop_duplicates().values)
    stream_df = stream_df.copy()
    stream_df["Driver"] = stream_df["Driver"].map(num_to_abbr)
    unmapped = stream_df["Driver"].isna().sum()
    if unmapped:
        print(f"    WARNING: {unmapped}/{len(stream_df)} stream_data rows had a car number "
              f"not found in this race's laps_data — dropping them")
        stream_df = stream_df.dropna(subset=["Driver"])
    return stream_df

def build_grid(laps_df, freq=GRID_FREQ):
    grids = []
    for driver, laps in laps_df.groupby("Driver"):
        start = laps["LapStartTime"].min()
        end = laps["Time"].max()
        if pd.isna(start) or pd.isna(end) or end <= start:
            print(f"    WARNING: skipping {driver}, bad bounds (start={start}, end={end})")
            continue
        times = pd.timedelta_range(start=start, end=end, freq=freq)
        grids.append(pd.DataFrame({"Driver": driver, "Time": times}))
    if not grids:
        raise RuntimeError("no drivers produced a usable grid at all")
    return pd.concat(grids, ignore_index=True).sort_values(["Driver", "Time"]).reset_index(drop=True)

def stitch_race(name):
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    laps_df = pd.read_parquet(RAW_DIR / f"{name}_laps.parquet")
    stream_df = pd.read_parquet(RAW_DIR / f"{name}_stream.parquet")
    car_df = pd.read_parquet(RAW_DIR / f"{name}_car.parquet")

    print(f"[load] laps: {len(laps_df)} rows, stream: {len(stream_df)} rows, car: {len(car_df)} rows")
    print(f"       drivers -> laps: {laps_df['Driver'].nunique()}, stream (raw): {stream_df['Driver'].nunique()}, car: {car_df['Driver'].nunique()}")

    # flag any driver present in car/stream but missing from laps entirely (the "No lap data
    # for driver X" scenario) - these drivers should just be silently absent from the grid,
    # not crash anything. Confirm that's actually what happens.
    laps_drivers = set(laps_df["Driver"].unique())
    car_drivers = set(car_df["Driver"].unique())
    missing_from_laps = car_drivers - laps_drivers
    if missing_from_laps:
        print(f"[check] driver(s) with car telemetry but NO laps_data at all: {missing_from_laps} "
              f"-> expected to be silently absent from the grid (build_grid groups by laps_df's Driver)")

    stream_df = fix_stream_driver_ids(stream_df, laps_df)
    print(f"       stream drivers after remap: {stream_df['Driver'].nunique()}")

    grid = build_grid(laps_df)
    print(f"[grid] {len(grid)} rows, {grid['Driver'].nunique()} drivers "
          f"(laps_data had {laps_df['Driver'].nunique()} drivers)")
    if grid["Driver"].nunique() != laps_df["Driver"].nunique():
        print("       NOTE: driver count mismatch between laps_data and the built grid - "
              "check the 'bad bounds' warnings above for why")

    span = grid.groupby("Driver")["Time"].agg(["min", "max", "count"])
    session_end = span["max"].max()
    early_cutoff = session_end - pd.Timedelta(minutes=5)
    retired = span[span["max"] < early_cutoff]
    if len(retired):
        print(f"[check] retirement-cutoff: {len(retired)} driver(s) whose grid ends >5min before session end:")
        print(retired)
    else:
        print("[check] retirement-cutoff: no driver's grid ends notably early")

    stream_df = stream_df.sort_values(["Driver", "Time"]).reset_index(drop=True)
    stream_df = clean_gap_column(stream_df, "IntervalToPositionAhead", "gap_ahead")
    stream_df = clean_gap_column(stream_df, "GapToLeader", "gap_leader")

    stream_cols = ["Driver", "Time", "Position", "gap_ahead", "gap_ahead_is_lapped", "gap_leader", "gap_leader_is_lapped"]
    stitched = pd.merge_asof(grid.sort_values("Time"), stream_df[stream_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    car_cols = ["Driver", "Time", "Speed", "DRS"]
    stitched = pd.merge_asof(stitched.sort_values("Time"), car_df[car_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    laps_join = laps_df.rename(columns={"Time": "_LapEndTime", "LapStartTime": "Time"})
    laps_cols = ["Driver", "Time", "LapNumber", "Compound", "TyreLife", "Stint", "PitInTime", "PitOutTime", "TrackStatus"]
    stitched = pd.merge_asof(stitched.sort_values("Time"), laps_join[laps_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    print(f"[stitch] final table: {len(stitched)} rows, {len(stitched.columns)} columns")

    for col in ["gap_ahead", "gap_leader", "Position", "Speed"]:
        n_null = stitched[col].isna().sum()
        pct = n_null / len(stitched) * 100
        flag = "  <-- INVESTIGATE" if pct > 5 else ""
        print(f"[check] {col} NaN: {n_null}/{len(stitched)} ({pct:.1f}%){flag}")

    # check for any Time ordering / duplicate-timestamp weirdness a lap-alignment
    # problem could plausibly produce
    dupe_check = stitched.groupby(["Driver", "Time"]).size()
    dupes = dupe_check[dupe_check > 1]
    if len(dupes):
        print(f"[check] WARNING: {len(dupes)} (Driver, Time) combinations appear more than once in the grid - unexpected")
    else:
        print("[check] no duplicate (Driver, Time) grid rows - OK")

    # time-reference spot check on the driver most likely to be affected, if known
    sample_driver = laps_df["Driver"].iloc[0]
    driver_laps = laps_df[laps_df["Driver"] == sample_driver].dropna(subset=["LapStartTime", "Time"]).reset_index(drop=True)
    if len(driver_laps):
        sample_lap = driver_laps.iloc[len(driver_laps) // 2]
        lap_start, lap_end = sample_lap["LapStartTime"], sample_lap["Time"]
        window = stitched[(stitched["Driver"] == sample_driver) & (stitched["Time"] >= lap_start) & (stitched["Time"] <= lap_end)]
        print(f"[check] time-reference spot check - {sample_driver}, lap {sample_lap['LapNumber']}, {lap_start} -> {lap_end}:")
        print(window[["Time", "Speed", "DRS", "gap_ahead"]].to_string(index=False))

    out_path = STITCHED_DIR / f"{name}.parquet"
    stitched.to_parquet(out_path)
    print(f"[save] {out_path} ({out_path.stat().st_size:,} bytes)")
    return stitched

results = {}
for name in RACES:
    try:
        results[name] = stitch_race(name)
    except Exception:
        print(f"\nFAILED on {name}")
        traceback.print_exc()

print("\n" + "=" * 70)
print(f"Done. {len(results)}/{len(RACES)} races stitched successfully.")
print("=" * 70)

python : d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Scripts\python.exe
pandas : 2.3.3

2021_03_portuguese_grand_prix_R
[load] laps: 1245 rows, stream: 28712 rows, car: 658360 rows
       drivers -> laps: 20, stream (raw): 20, car: 20
       stream drivers after remap: 20
[grid] 21766 rows, 20 drivers (laps_data had 20 drivers)
[check] retirement-cutoff: 1 driver(s) whose grid ends >5min before session end:
                          min                    max  count
Driver                                                     
RAI    0 days 00:33:02.841000 0 days 00:37:02.841000     49


C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\ipykernel_4832\2985673199.py:117: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  early_cutoff = session_end - pd.Timedelta(minutes=5)


[stitch] final table: 21766 rows, 16 columns
[check] gap_ahead NaN: 36/21766 (0.2%)
[check] gap_leader NaN: 36/21766 (0.2%)
[check] Position NaN: 0/21766 (0.0%)
[check] Speed NaN: 0/21766 (0.0%)
[check] no duplicate (Driver, Time) grid rows - OK
[check] time-reference spot check - GAS, lap 34.0, 0 days 01:23:17.133000 -> 0 days 01:24:41.254000:
                  Time  Speed  DRS  gap_ahead
0 days 01:23:17.841000  296.0    0      2.843
0 days 01:23:22.841000  298.0    0      2.883
0 days 01:23:27.841000  179.0    0      0.086
0 days 01:23:32.841000   82.0    0      0.744
0 days 01:23:37.841000  183.0    0      1.024
0 days 01:23:42.841000  282.0    0      1.118
0 days 01:23:47.841000   71.0    0      1.087
0 days 01:23:52.841000  236.0    0      1.088
0 days 01:23:57.841000  158.0    0      1.054
0 days 01:24:02.841000  187.0    0      1.036
0 days 01:24:07.841000  277.0    0      1.213
0 days 01:24:12.841000  128.0    0      1.185
0 days 01:24:17.841000  255.0    0      1.465
0 days 01

C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\ipykernel_4832\2985673199.py:117: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  early_cutoff = session_end - pd.Timedelta(minutes=5)


[stitch] final table: 18470 rows, 16 columns
[check] gap_ahead NaN: 39/18470 (0.2%)
[check] gap_leader NaN: 38/18470 (0.2%)
[check] Position NaN: 0/18470 (0.0%)
[check] Speed NaN: 0/18470 (0.0%)
[check] no duplicate (Driver, Time) grid rows - OK
[check] time-reference spot check - VER, lap 36.0, 0 days 01:14:07.694000 -> 0 days 01:15:16.425000:
                  Time  Speed  DRS  gap_ahead
0 days 01:14:10.677000  298.0    0        0.0
0 days 01:14:15.677000  171.0    0        0.0
0 days 01:14:20.677000  283.0    0        0.0
0 days 01:14:25.677000  280.0    0        0.0
0 days 01:14:30.677000   93.0    0        0.0
0 days 01:14:35.677000  267.0    0        0.0
0 days 01:14:40.677000  256.0    0        0.0
0 days 01:14:45.677000  177.0    0        0.0
0 days 01:14:50.677000  243.0    0        0.0
0 days 01:14:55.677000  239.0    0        0.0
0 days 01:15:00.677000  250.0    0        0.0
0 days 01:15:05.677000  292.0    0        0.0
0 days 01:15:10.677000  215.0    0        0.0
0 days 01

C:\Users\MEGA IT SOLUTION\AppData\Local\Temp\ipykernel_4832\2985673199.py:117: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  early_cutoff = session_end - pd.Timedelta(minutes=5)


[stitch] final table: 16896 rows, 16 columns
[check] gap_ahead NaN: 28/16896 (0.2%)
[check] gap_leader NaN: 27/16896 (0.2%)
[check] Position NaN: 0/16896 (0.0%)
[check] Speed NaN: 0/16896 (0.0%)
[check] no duplicate (Driver, Time) grid rows - OK
[check] time-reference spot check - VER, lap 26.0, 0 days 01:46:45.247000 -> 0 days 01:48:18.551000:
                  Time  Speed  DRS  gap_ahead
0 days 01:46:47.563000  303.0    0      1.187
0 days 01:46:52.563000  106.0    0      1.047
0 days 01:46:57.563000  188.0    0      1.088
0 days 01:47:02.563000  173.0    0      1.152
0 days 01:47:07.563000  231.0    0      1.250
0 days 01:47:12.563000  217.0    0      1.375
0 days 01:47:17.563000  265.0    0      1.516
0 days 01:47:22.563000  257.0    0      1.514
0 days 01:47:27.563000  183.0    0      1.403
0 days 01:47:32.563000  276.0    0      1.494
0 days 01:47:37.563000  201.0    0      1.432
0 days 01:47:42.563000  285.0    0      1.523
0 days 01:47:47.563000  309.0    0      1.481
0 days 01

In [5]:
# =========================================================
# Stage 2 + Stage 3 — FULL SCALE-UP
# Runs the validated stitching pipeline (grid + merge_asof, with the
# stream_data car-number->abbreviation fix) across every race found in
# data/raw/, skipping anything already in data/stitched/.
#
# Race list is DISCOVERED from data/raw/, not hardcoded — this means the
# 6 sessions excluded from the training scope (2023 Round 2 skip, plus
# Rounds 3/4R/4S/5/6 never saved) are automatically absent and simply
# never attempted, no exclusion list needed. Confirmed on 5 races so far
# (2021 Bahrain, Imola, Portugal, Styria; 2022 Saudi Arabia) covering both
# the normal case and the lap-alignment-warning edge case.
# =========================================================

import re
import sys
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

print(f"python : {sys.executable}")
print(f"pandas : {pd.__version__}")
if int(pd.__version__.split(".")[0]) >= 3:
    raise RuntimeError(f"pandas {pd.__version__} >= 3.0, wrong kernel?")

RAW_DIR = Path("data") / "raw"
STITCHED_DIR = Path("data") / "stitched"
STITCHED_DIR.mkdir(parents=True, exist_ok=True)

GRID_FREQ = "5s"
GAP_NULL_WARN_PCT = 5.0       # flag in the summary if gap_ahead null% exceeds this
CONSECUTIVE_FAILURE_LIMIT = 3  # circuit breaker, matches the Stage 1 loader's pattern

# ---------- discover races from data/raw/ ----------
def discover_races(raw_dir):
    """A race is only included if it has laps + stream + car files (the three
    this pipeline actually reads). Missing/partial races (e.g. the old-format
    2023_01_bahrain_R duplicate with only 2 files) are silently skipped."""
    stems = set()
    for p in raw_dir.glob("*_laps.parquet"):
        stems.add(p.name[: -len("_laps.parquet")])
    races = []
    incomplete = []
    for stem in sorted(stems):
        needed = [raw_dir / f"{stem}_laps.parquet", raw_dir / f"{stem}_stream.parquet", raw_dir / f"{stem}_car.parquet"]
        if all(f.exists() for f in needed):
            races.append(stem)
        else:
            incomplete.append(stem)
    return races, incomplete

# ---------- same fixed Stage 2/3 logic, validated on 5 races ----------
def parse_gap_value(raw, position):
    if pd.isna(raw):
        return np.nan, "missing"
    s = str(raw).strip()
    if re.fullmatch(r"[+-]\d+\.?\d*", s):
        return float(s), "real"
    if re.fullmatch(r"LAP \d+", s):
        return (0.0 if position == 1 else np.nan), "lap_marker"
    if re.fullmatch(r"\d+ L", s):
        return np.nan, "lapped"
    if re.fullmatch(r"\d+L", s):
        return np.nan, "malformed"
    return np.nan, "malformed"

def clean_gap_column(df, raw_col, out_col):
    parsed = df.apply(lambda r: parse_gap_value(r[raw_col], r["Position"]), axis=1)
    df[out_col] = [p[0] for p in parsed]
    df[f"{out_col}_kind"] = [p[1] for p in parsed]
    df[f"{out_col}_is_lapped"] = df[f"{out_col}_kind"] == "lapped"
    df[out_col] = df.groupby("Driver")[out_col].ffill()
    return df

def fix_stream_driver_ids(stream_df, laps_df):
    num_to_abbr = dict(laps_df[["DriverNumber", "Driver"]].drop_duplicates().values)
    stream_df = stream_df.copy()
    stream_df["Driver"] = stream_df["Driver"].map(num_to_abbr)
    stream_df = stream_df.dropna(subset=["Driver"])
    return stream_df

def build_grid(laps_df, freq=GRID_FREQ):
    grids = []
    for driver, laps in laps_df.groupby("Driver"):
        start = laps["LapStartTime"].min()
        end = laps["Time"].max()
        if pd.isna(start) or pd.isna(end) or end <= start:
            continue
        times = pd.timedelta_range(start=start, end=end, freq=freq)
        grids.append(pd.DataFrame({"Driver": driver, "Time": times}))
    if not grids:
        raise RuntimeError("no drivers produced a usable grid at all")
    return pd.concat(grids, ignore_index=True).sort_values(["Driver", "Time"]).reset_index(drop=True)

def stitch_race(name):
    laps_df = pd.read_parquet(RAW_DIR / f"{name}_laps.parquet")
    stream_df = pd.read_parquet(RAW_DIR / f"{name}_stream.parquet")
    car_df = pd.read_parquet(RAW_DIR / f"{name}_car.parquet")

    stream_df = fix_stream_driver_ids(stream_df, laps_df)
    grid = build_grid(laps_df)

    stream_df = stream_df.sort_values(["Driver", "Time"]).reset_index(drop=True)
    stream_df = clean_gap_column(stream_df, "IntervalToPositionAhead", "gap_ahead")
    stream_df = clean_gap_column(stream_df, "GapToLeader", "gap_leader")

    stream_cols = ["Driver", "Time", "Position", "gap_ahead", "gap_ahead_is_lapped", "gap_leader", "gap_leader_is_lapped"]
    stitched = pd.merge_asof(grid.sort_values("Time"), stream_df[stream_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    car_cols = ["Driver", "Time", "Speed", "DRS"]
    stitched = pd.merge_asof(stitched.sort_values("Time"), car_df[car_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    laps_join = laps_df.rename(columns={"Time": "_LapEndTime", "LapStartTime": "Time"})
    laps_cols = ["Driver", "Time", "LapNumber", "Compound", "TyreLife", "Stint", "PitInTime", "PitOutTime", "TrackStatus"]
    stitched = pd.merge_asof(stitched.sort_values("Time"), laps_join[laps_cols].sort_values("Time"),
                              on="Time", by="Driver", direction="backward")

    return stitched

# ---------- run ----------
races, incomplete = discover_races(RAW_DIR)
print(f"discovered {len(races)} races with a complete raw triple (laps+stream+car)")
if incomplete:
    print(f"skipping {len(incomplete)} incomplete race(s), missing one or more of laps/stream/car: {incomplete}")

already_done = {p.stem for p in STITCHED_DIR.glob("*.parquet")}
todo = [r for r in races if r not in already_done]
print(f"{len(already_done)} already stitched, {len(todo)} remaining to process\n")

summary = {"ok": [], "failed": [], "high_null": []}
consecutive_failures = 0

for i, name in enumerate(todo, 1):
    t0 = time.time()
    try:
        stitched = stitch_race(name)
        out_path = STITCHED_DIR / f"{name}.parquet"
        stitched.to_parquet(out_path)
        dt = time.time() - t0
        gap_null_pct = stitched["gap_ahead"].isna().mean() * 100
        n_rows = len(stitched)
        n_drivers = stitched["Driver"].nunique()
        flag = ""
        if gap_null_pct > GAP_NULL_WARN_PCT:
            flag = f"  <-- gap_ahead null {gap_null_pct:.1f}%, investigate"
            summary["high_null"].append((name, gap_null_pct))
        print(f"[{i}/{len(todo)}] OK   {name}  ({n_rows} rows, {n_drivers} drivers, {dt:.1f}s){flag}")
        summary["ok"].append(name)
        consecutive_failures = 0
    except Exception as e:
        dt = time.time() - t0
        print(f"[{i}/{len(todo)}] FAIL {name}  ({dt:.1f}s)")
        traceback.print_exc()
        summary["failed"].append((name, str(e)))
        consecutive_failures += 1
        if consecutive_failures >= CONSECUTIVE_FAILURE_LIMIT:
            print(f"\nSTOPPING: {CONSECUTIVE_FAILURE_LIMIT} consecutive failures — likely an environment "
                  f"problem, not per-race data issues. Fix and rerun; already-stitched races are skipped "
                  f"automatically next time.")
            break

print("\n" + "=" * 70)
print(f"Done. {len(summary['ok'])} succeeded, {len(summary['failed'])} failed, "
      f"{len(summary['high_null'])} flagged for high gap_ahead null%.")
if summary["failed"]:
    print("\nFailed races:")
    for name, err in summary["failed"]:
        print(f"  {name}: {err}")
if summary["high_null"]:
    print("\nHigh null% races (worth a manual look before trusting):")
    for name, pct in summary["high_null"]:
        print(f"  {name}: {pct:.1f}%")
print("=" * 70)

python : d:\Personal\Projects\F1 Multiview with Ml feed Recomender\.venv\Scripts\python.exe
pandas : 2.3.3
discovered 132 races with a complete raw triple (laps+stream+car)
skipping 1 incomplete race(s), missing one or more of laps/stream/car: ['2023_01_bahrain_R']
5 already stitched, 127 remaining to process

[1/127] OK   2021_04_spanish_grand_prix_R  (21553 rows, 20 drivers, 1.3s)
[2/127] OK   2021_05_monaco_grand_prix_R  (21992 rows, 19 drivers, 0.9s)
[3/127] OK   2021_06_azerbaijan_grand_prix_R  (29092 rows, 20 drivers, 0.9s)
[4/127] OK   2021_07_french_grand_prix_R  (21159 rows, 20 drivers, 0.9s)
[5/127] OK   2021_09_austrian_grand_prix_R  (19303 rows, 20 drivers, 0.8s)
[6/127] OK   2021_10_british_grand_prix_R  (26964 rows, 20 drivers, 0.9s)
[7/127] OK   2021_10_british_grand_prix_S  (6312 rows, 20 drivers, 0.4s)
[8/127] OK   2021_11_hungarian_grand_prix_R  (21610 rows, 20 drivers, 1.1s)
[9/127] OK   2021_12_belgian_grand_prix_R  (2580 rows, 20 drivers, 0.9s)  <-- gap_ahead null 